## import libraries

In [ ]:
# ── Standard library ──────────────────────────────────────────
import os
import sys
import time
import json
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict

# Data handling
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

# Image processing & visualization
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Detection (YOLOv8)
import ultralytics
from ultralytics import YOLO

# import cfgs
PROJECT_ROOT = "/home/user/NeuroVision-AI-2"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = PROJECT_ROOT

from src.config import cfg

#  Quick check
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("ultralytics :", ultralytics.__version__)
print("opencv      :", cv2.__version__)

print("Raw data :", cfg.RAW_DIR)
print("Classes  :", cfg.CLASS_NAMES)
print("Split    : train %.0f%% / val %.0f%% / test %.0f%%" % (
    (1 - cfg.VAL_RATIO - cfg.TEST_RATIO) * 100, cfg.VAL_RATIO * 100, cfg.TEST_RATIO * 100))
print("Seed     :", cfg.SEED)

## download data

In [ ]:
import os
import sys

PROJECT_ROOT = "/home/user/NeuroVision-AI-2"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = PROJECT_ROOT

from src.dataset import download_dataset

download_dataset()

## Organizing images/labels into train, valid, test

In [ ]:
from src.dataset import collect_pairs

pairs, orphans = collect_pairs()
print("Pairs per class:", {c: len(v) for c, v in pairs.items()})
print("Total pairs    :", sum(len(v) for v in pairs.values()))
print("Orphan images  :", len(orphans))

## Collect image/label pairs and count orphans

In [ ]:
from src.dataset import collect_pairs

pairs, orphans = collect_pairs()
print("Pairs per class:", {c: len(v) for c, v in pairs.items()})
print("Total pairs    :", sum(len(v) for v in pairs.values()))
print("Orphan images  :", len(orphans))

## Read a YOLO annotation and convert it to pixels

In [ ]:
# One annotation, normalized then in pixels
from src.annotations import show_annotation

image_path, label_path = next((i, l) for items in pairs.values() for i, l in items
                              if l.read_text().strip())
show_annotation(image_path, label_path, class_names=cfg.CLASS_NAMES)

## Show 9 images with boxes and class names

In [ ]:
from src.visualization import show_samples_grid

show_samples_grid(pairs, n=9, seed=cfg.SEED, class_names=cfg.CLASS_NAMES)

## Class distribution

In [ ]:
# Cell 7: Distribution
from src.dataset import class_distribution
from src.visualization import plot_class_distribution

dist = class_distribution(pairs)   # DataFrame: class, images, boxes
display(dist)
plot_class_distribution(dist)
print("Imbalance ratio (largest / smallest class): %.2f" % (dist.images.max() / dist.images.min()))

## Cleaning and validation audit

In [ ]:
from src.dataset import audit_dataset

audit = audit_dataset(pairs)
print(json.dumps({k: len(v) for k, v in audit.items()}, indent=2))

## Split 70/15/15 and organize in YOLO format

In [ ]:
from src.dataset import split_pairs, organize

counts = split_pairs(pairs)
report = organize()
print(json.dumps(report, indent=2))

## Count per split and write `data.yaml`

In [ ]:
from src.dataset import class_id_map, count_split, write_data_yaml, validate_data_yaml

for folder, ids in class_id_map().items():
    print(f"{folder:12s} -> class ids in labels: {dict(ids)}")
print("cfg.CLASS_NAMES:", dict(enumerate(cfg.CLASS_NAMES)), "\n")

for split, (n_img, n_lbl, match) in count_split().items():
    print(f"{split:6s} images={n_img:5d}  labels={n_lbl:5d}  match={match}")

write_data_yaml()
print("\n" + cfg.DATA_YAML.read_text())
print("Problems:", validate_data_yaml() or "none, data.yaml is valid")

## Leakage and class balance check

In [ ]:
from src.dataset import check_leakage

balance, leakage = check_leakage(class_names=cfg.CLASS_NAMES)
print("Share of each class per split (%):")
display(balance)
print("\nOverlap between splits (must be 0):")
display(leakage)